In [23]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd
import requests
import io
import time
import csv
from geopy.geocoders import ArcGIS
from geopy.exc import GeocoderTimedOut

## 門牌地址定位服務

In [24]:
def get_coords_from_arcgis(address, geolocator):
    """利用 ArcGIS API 將台灣地址轉為經緯度，並加入重試機制"""
    # 清理地址格式
    address = str(address).replace('臺', '台').strip()
    # 確保只有一個「台中市」前綴
    if not address.startswith("台中市"):
        full_address = f"台中市{address}"
    else:
        full_address = address

    # 嘗試最多 3 次
    for attempt in range(3):
        try:
            location = geolocator.geocode(full_address)
            if location:
                return location.latitude, location.longitude
            return None, None
        except (GeocoderTimedOut, Exception):
            if attempt < 2:
                time.sleep(2)  # 發生錯誤時多等兩秒
                continue
            return None, None
    return None, None

In [57]:
def create_relax_map():
  # 1. 初始化地圖，中心點設在台中市中心 (草悟道附近)
  # taichung_coords = [24.1508, 120.6637]
  # m = folium.Map(location=taichung_coords, zoom_start=13, control_scale=True)

  # 使用 Google Maps Tiles 連結
  google_map_tiles = 'https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}'

  taichung_coords = [24.1508, 120.6637]
  m = folium.Map(
    location=taichung_coords,
    zoom_start=12,
    control_scale=True,
    tiles=google_map_tiles,
    attr='Google Maps'
  )

  # 初始化地理編碼器 (Nominatim 是免費的 OSM 服務)
  geolocator = ArcGIS(timeout=10)

  # 2. 取得正確的開放資料來源
    # 來源 A: 台中市視障按摩據點 (CSV 格式)
  massage_url = "https://newdatacenter.taichung.gov.tw/api/v1/no-auth/resource.download?rid=45b70e59-be63-49f7-ab39-785899026c63"
    # 來源 B: 台中市廁所
  toilet_url = "https://newdatacenter.taichung.gov.tw/api/v1/no-auth/resource.download?rid=d5f667b9-4583-4b72-95dc-355c85fd130f"

  # 3. 建立圖層組
    # 使用 MarkerCluster 處理按摩據點
  massage_cluster = MarkerCluster(name='按摩據點 (Relaxation)').add_to(m)
    # 使用 FeatureGroup 處理公廁
  toilet_group = folium.FeatureGroup(name='列管公廁 (Restroom)').add_to(m)

  # 4. 處理資料
    # 處理資料-「來源 A: 台中市視障按摩據點」

  # --- 處理按摩店資料 ---
  success_count = 0
  try:
    response = requests.get(massage_url)
    response.encoding = 'utf-8'
    #df_massage = pd.read_csv(io.StringIO(response.text))
    df_massage = pd.read_csv(
            io.StringIO(response.text),
            on_bad_lines='warn',
            engine='python',
            quoting=csv.QUOTE_MINIMAL
    )

    # 設定處理上限
    # process_limit = 2
    # print(f"開始轉換地址 (上限 {process_limit} 筆)...")

    # for index, item in df_massage.head(process_limit).iterrows():
    for index, item in df_massage.iterrows():
      name = item.get('視障按摩據點名稱', '未知據點')
      addr = item.get('地址', '')
      tel = item.get('聯絡電話', '無電話')

      lat, lon = get_coords_from_arcgis(addr, geolocator)

      if lat and lon:
        # 建立獨立的標註點對象
        popup_content = f"""
          <div style="font-family: 'Microsoft JhengHei'; width: 200px;">
          <h4 style="margin-bottom:5px;">{name}</h4>
          <p style="font-size:12px;"><b>地址:</b> {addr}</p>
          <p style="font-size:12px;"><b>電話:</b> {tel}</p>
          <a href="https://www.google.com/maps/search/?api=1&query={lat},{lon}" target="_blank">Google 導航</a>
          </div>
        """
        # 確保每一筆都建立一個新的 Marker 並加入 cluster
        marker = folium.Marker(
          location=[float(lat), float(lon)],
          popup=folium.Popup(popup_content, max_width=300),
          # icon=folium.Icon(color='purple', icon='hand-right', prefix='fa'),
          icon=folium.Icon(color='darkpurple', icon='certificate', prefix='fa'),
        )
        marker.add_to(massage_cluster)

        success_count += 1
        print(f"[{success_count}] 成功解析: {name} ({lat}, {lon})")
      else:
        print(f"[失敗] 無法解析地址: {addr}")

        # 增加延遲以防被 ArcGIS 封鎖
        time.sleep(1.2)

  except Exception as e:
        print(f"按摩資料處理發生錯誤: {e}")

  #   # --- 處理公廁資料 ---
  try:
    res_toilet = requests.get(toilet_url)
    res_toilet.encoding = 'utf-8'
    df_toliet = pd.read_csv(io.StringIO(res_toilet.text))

    # 移除地址重複的地點
    df_toliet = df_toliet.drop_duplicates(subset=['地址或地點描述'], keep='first')
    df_toliet = df_toliet.reset_index(drop=True)
    #df_toliet

    # 設定處理上限
    # process_limit = 2
    # print(f"開始轉換地址 (上限 {process_limit} 筆)...")

    # for index, item in df_toliet.head(process_limit).iterrows():
    for index, item in df_toliet.iterrows():
      name = item.get('公廁名稱', '未知據點')
      addr = item.get('地址或地點描述', '')
      caty = item.get('公廁類別', '未知')
      auth = item.get('權管單位', '未知')
      lat = item.get('緯度', '')
      lon = item.get('經度', '')
      if lat and lon:
        # name = item.get('公廁名稱') or item.get('名稱', '公廁')
        popup_content = f"""
          <div style="font-family: 'Microsoft JhengHei'; width: 200px;">
          <h4 style="margin-bottom:5px;">{name}</h4>
          <p style="font-size:12px;"><b>地址:</b> {auth}</p>
          <p style="font-size:12px;"><b>公廁類別:</b> {caty}</p>
          <p style="font-size:12px;"><b>權管單位:</b> {auth}</p>
          <a href="https://www.google.com/maps/search/?api=1&query={lat},{lon}" target="_blank">Google 導航</a>
          </div>
        """
        folium.Marker(
          location=[float(lat), float(lon)],
          # popup=f"{name}",
          popup=folium.Popup(popup_content, max_width=300),
          # icon=folium.Icon(color='blue', icon='info-sign'),
          icon=folium.Icon(color='blue', icon='user', prefix='fa'),
        ).add_to(toilet_group)
  except Exception as e:
    print(f"無法獲取公廁資料: {e}")


  # try:
  #     print("\n正在抓取公廁資料...")
      # res_toilet = requests.get(toilet_url)
  #     toilet_data = res_toilet.json()
  #     for item in toilet_data:
  #       lat = item.get('Latitude') or item.get('緯度')
  #       lon = item.get('Longitude') or item.get('經度')
  #       if lat and lon:
  #         name = item.get('公廁名稱') or item.get('名稱', '公廁')
  #         folium.Marker(
  #           location=[float(lat), float(lon)],
  #           popup=f"{name}",
  #           icon=folium.Icon(color='blue', icon='info-sign'),
  #       ).add_to(toilet_group)
  # except Exception as e:
  #       print(f"無法獲取公廁資料: {e}")

  # 4. 圖層控制 (LayerControl 放在最後)
  folium.LayerControl().add_to(m)

  # 5. 儲存地圖
  m.save("taichung_relax_map.html")
  print(f"\n任務完成！地圖已儲存為 taichung_relax_map.html")
  print(f"按摩據點成功標註總數: {success_count}")

if __name__ == "__main__":
    create_relax_map()

/tmp/ipykernel_220/2418701547.py:42: ParserWarning: Skipping line 105: unexpected end of data

  df_massage = pd.read_csv(


[1] 成功解析: 204專業視障按摩 (24.254048994861, 120.722935991698)
[2] 成功解析: 上好盲人專業按摩 (24.160545805544, 120.663796803292)
[3] 成功解析: 上安視障按摩 (24.307803983943, 120.724702980707)
[4] 成功解析: 大台灣按摩中心 (24.133516011822, 120.719368569888)
[5] 成功解析: 大忙人視障按摩 (24.159556992426, 120.693118041704)
[6] 成功解析: 大肚按摩站 (24.155879013313, 120.543878004208)
[7] 成功解析: 大拇師傅視障按摩 (24.1846060167, 120.5852079953)
[8] 成功解析: 大蒼視障按摩 (24.346961732788, 120.617001109415)
[9] 成功解析: 中市盲協視障按摩便利站（中國急重樓站） (24.153570008536, 120.682712998559)
[10] 成功解析: 中市盲協視障按摩便利站（慈濟大愛站） (24.195685006769, 120.721944999286)
[11] 成功解析: 中市盲協視障按摩便利站（榮總新門診站） (24.183498180557, 120.603714818416)
[12] 成功解析: 中華啟明視障按摩便利站-家樂福西屯店 (24.183939110573, 120.61473283014)
[13] 成功解析: 今日按摩中心（光復店） (24.142419981837, 120.682786004936)
[14] 成功解析: 今日按摩中心（遼陽店） (24.171810915774, 120.693086861024)
[15] 成功解析: 心光精緻按摩館 (24.170057002534, 120.646734014455)
[16] 成功解析: 文勝按摩健康中心 (24.244919384012, 120.701513858839)
[17] 成功解析: 巧手視障按摩工作室 (24.171582215545, 120.663058860536)
[18] 成功解析: 弘安視障按摩養生中心 